In [24]:
import joblib
import pandas as pd

df = pd.read_csv(r'C:\Users\win10\OneDrive\Desktop\Data-Projects\Portfolio Project\HR Attrition Analytics & Prediction\01_Data\01_Raw_Data/cleaned_hr_data.csv')

In [25]:
# Load Encoder 
model = joblib.load(r'C:\Users\win10\OneDrive\Desktop\Data-Projects\Portfolio Project\HR Attrition Analytics & Prediction\04_Model\attrition_model.pkl')

In [26]:
employee_ids = df['EmployeeNumber']

In [27]:
# Create Target Columns 
df['AttritionFlag'] = df['Attrition'].map({'Yes':1, 'No':0})

In [28]:
drop_cols = [
    'Attrition',
    'EmployeeCount',
    'EmployeeNumber',
    'Over18',
    'StandardHours'
]

df = df.drop(columns=drop_cols)

In [29]:
X = df.drop('AttritionFlag', axis=1)

In [34]:
# Apply Saved Encoders
encoders = joblib.load(r'C:\Users\win10\OneDrive\Desktop\Data-Projects\Portfolio Project\HR Attrition Analytics & Prediction\04_Model\attrition_model.pkl')
categorical_cols = X.select_dtypes(include='object').columns

for col in categorical_cols:
    X[col] = encoders[col].transform(
        X[col]
    )

In [31]:
print(X.shape)

(1470, 31)


In [35]:
X.shape[1]

31

In [36]:
# Generate Risk Score 
risk_score = model.predict_proba(X)[:,1]
risk_score

array([0.73      , 0.04333333, 0.45333333, ..., 0.14333333, 0.13666667,
       0.08      ], shape=(1470,))

In [38]:
results = pd.read_csv(r'C:\Users\win10\OneDrive\Desktop\Data-Projects\Portfolio Project\HR Attrition Analytics & Prediction\01_Data\01_Raw_Data\Cleaned_HR_Data.csv')
results['RiskScore'] = risk_score

In [39]:
# Create Risk Category 
results['RiskLevel'] = pd.cut(results['RiskScore'], bins=[0,0.3,0.6,1],labels=['Low','Medium','High'])

In [43]:
import numpy as np

# Create HR Priority Levels 
results['HR_Action'] = np.where(
    results['RiskScore'] >= 0.7,
    'Immediate Intervention',
    np.where(
        results['RiskScore'] >= 0.5,
        'Monitor',
        'Stable'
    )
)
results['HR_Action']

0       Immediate Intervention
1                       Stable
2                       Stable
3                       Stable
4                       Stable
                 ...          
1465                    Stable
1466                    Stable
1467                    Stable
1468                    Stable
1469                    Stable
Name: HR_Action, Length: 1470, dtype: str

In [44]:
# Top Risk Employees
high_risk = results.sort_values(by='RiskScore', ascending=False)

high_risk[[
        'EmployeeNumber',
        'Department',
        'JobRole',
        'MonthlyIncome',
        'RiskScore',
        'RiskLevel'
    ]].head(20)

,EmployeeNumber,Department,JobRole,MonthlyIncome,RiskScore,RiskLevel
463,622,Research & Development,Laboratory Technician,2340,0.966667,High
1332,1868,Research & Development,Research Scientist,2439,0.926667,High
127,167,Sales,Sales Representative,1675,0.923333,High
683,952,Sales,Sales Representative,2413,0.916667,High
1016,1433,Research & Development,Research Scientist,1261,0.913333,High
1339,1878,Research & Development,Research Scientist,2472,0.906667,High
892,1248,Research & Development,Research Scientist,1859,0.906667,High
1012,1427,Sales,Sales Representative,1359,0.906667,High
688,959,Sales,Sales Representative,2121,0.896667,High
1136,1604,Research & Development,Laboratory Technician,2408,0.886667,High


In [45]:
# Risk Distribution 
results['RiskLevel'].value_counts()

RiskLevel
Low       1230
High       194
Medium      40
Name: count, dtype: int64

In [46]:
# Department Risk Analysis 
results.groupby('Department')['RiskScore'].mean().sort_values(ascending=False)

Department
Sales                     0.201472
Human Resources           0.193386
Research & Development    0.150697
Name: RiskScore, dtype: float64

In [47]:
results.to_csv(
   r"C:\Users\win10\OneDrive\Desktop\Data-Projects\Portfolio Project\HR Attrition Analytics & Prediction\01_Data\02_Processed_Data\employee_attrition_risk.csv",
    index=False
)